# Example 6.5.1 / 2: Solve a Named Problem from amplpy

Companion notebook for **6.5.1 Named Problems and Environments** in the AMPL lab Documentation, section "Solve a Named Problem from amplpy". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a Named Problem from amplpy

`ampl.solve()` accepts a `problem=` argument, but it does not combine safely with `solver=`: the
solver option is set in the environment that is current before the call, and the solve then
switches to the named problem's own environment, where no solver is set. With the two problems
from the first example, `B` is current after the model is read:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 0;                    # first decision, nonnegative
var y >= 0;                    # second decision, nonnegative
minimize ObjA: x + 2*y;        # objective of problem A
minimize ObjB: 3*x + y;        # objective of problem B
subject to Link: x + y >= 10;  # x + y at least 10, in both problems

problem A: x, y, ObjA, Link;  # gets its own environment A
problem B: x, y, ObjB, Link;  # gets environment B and becomes current

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(problem="A", solver="highs")

Error executing "solve" command:
No solver specified:  option solver is ''.


### Another way in amplpy

Select the problem first, then set options and solve, so that they go to its environment:

In [5]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [6]:
# run.py
ampl.eval("problem A;")
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.var["x"].value(), ampl.var["y"].value())

solved 10.0 0.0


### Pure AMPL: model.run

The same failure in pure AMPL: `solve A;` selects `A` before it solves, and the solver was set
only in `B`'s environment:

In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve A;

Error executing "solve" command:
No solver specified:  option solver is ''.
